# 02 Case Representation

Notebook ini digunakan untuk tahap kedua sistem Case-Based Reasoning (CBR), yaitu merepresentasikan setiap putusan ke dalam bentuk data terstruktur.

Input yang digunakan:
- File hasil tahap 1: `stage_01_build_case_base_output.zip`
- Folder teks putusan: `data/text/`

Output utama:
- `data/processed/cases.csv`
- `data/processed/cases.json`
- `data/processed/metadata_quality_report.csv`

Kolom utama yang dibuat:
- `case_id`
- `nomor_perkara`
- `tanggal_putusan`
- `terdakwa`
- `pengadilan`
- `jenis_perkara`
- `pasal`
- `amar_putusan`
- `ringkasan_fakta`
- `word_count`
- `text_full`


## 1. Instalasi Library

Cell ini memasang library yang dibutuhkan untuk membaca file, mengolah teks, membuat tabel, dan menyimpan hasil representasi kasus.


In [ ]:
!pip -q install pandas tqdm openpyxl

## 2. Import Library

Cell ini memanggil library Python yang digunakan untuk ekstraksi metadata, pemrosesan teks, validasi data, dan penyimpanan file CSV/JSON.


In [ ]:
import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

print("Library berhasil dimuat.")

## 3. Menyiapkan Struktur Folder

Cell ini menyiapkan struktur folder project. Jika hasil tahap 1 sudah diekstrak, folder `data/text/` akan digunakan sebagai sumber teks putusan.


In [ ]:
BASE_DIR = Path("/content/cbr-putusan-penganiayaan")

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder project siap digunakan.")
print(BASE_DIR)

## 4. Upload dan Ekstrak Output Tahap 1

Cell ini meminta file `stage_01_build_case_base_output.zip` dari tahap pertama. File ZIP tersebut berisi hasil ekstraksi PDF menjadi teks yang akan digunakan untuk membuat representasi kasus.


In [ ]:
EXPECTED_ZIP_NAME = "stage_01_build_case_base_output.zip"
zip_path = Path("/content") / EXPECTED_ZIP_NAME

if not zip_path.exists():
    try:
        from google.colab import files
        print("Silakan upload file stage_01_build_case_base_output.zip")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        zip_path = Path("/content") / list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            "File ZIP tahap 1 belum tersedia. Upload file stage_01_build_case_base_output.zip terlebih dahulu."
        ) from e

print(f"File ZIP digunakan: {zip_path}")

# Ekstrak ZIP tahap 1
extract_temp = Path("/content/stage_01_extract_temp")
if extract_temp.exists():
    shutil.rmtree(extract_temp)
extract_temp.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_temp)

# Deteksi apakah ZIP berisi folder data langsung atau folder project di dalamnya
candidate_data_dirs = list(extract_temp.rglob("data"))

if len(candidate_data_dirs) == 0:
    raise FileNotFoundError("Folder data tidak ditemukan di dalam ZIP tahap 1.")

source_root = candidate_data_dirs[0].parent

# Salin hasil ekstraksi ke BASE_DIR
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
shutil.copytree(source_root, BASE_DIR)

# Definisikan ulang path setelah copy
RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

txt_files = sorted(TEXT_DIR.glob("*.txt"))

print(f"Jumlah file TXT ditemukan: {len(txt_files)}")
if len(txt_files) != 30:
    print("PERINGATAN: Jumlah file TXT bukan 30. Periksa kembali hasil tahap 1.")

## 5. Fungsi Dasar untuk Normalisasi Teks

Cell ini berisi fungsi pembantu untuk merapikan hasil ekstraksi metadata, menghapus spasi berlebih, dan menghitung jumlah kata.


In [ ]:
def normalize_space(text):
    """
    Merapikan spasi dan newline pada teks.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def one_line(text):
    """
    Mengubah teks beberapa baris menjadi satu baris yang rapi.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip(" :;-—\n\t")


def count_words(text):
    """
    Menghitung jumlah kata pada teks.
    """
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b\w+\b", text))


def clean_case_value(value):
    """
    Membersihkan nilai hasil ekstraksi agar lebih rapi.
    """
    value = one_line(value)
    value = value.replace("  ", " ")
    return value.strip()


## 6. Fungsi Ekstraksi Metadata

Cell ini berisi fungsi untuk mengambil metadata utama dari teks putusan, seperti nomor perkara, nama terdakwa, pasal, tanggal putusan, amar putusan, dan ringkasan fakta.


In [ ]:
def extract_nomor_perkara(text):
    patterns = [
        r"(?:P\s*U\s*T\s*U\s*S\s*A\s*N|PUTUSAN)\s*Nomor\s+([0-9]+/Pid\.B/[0-9]+/PN\s*Rap)",
        r"Nomor\s+([0-9]+/Pid\.B/[0-9]+/PN\s*Rap)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return clean_case_value(match.group(1)).replace("PN Rap", "PN Rap")

    return ""


def extract_terdakwa(text):
    patterns = [
        r"Nama lengkap\s*:\s*(.*?)(?:\n\s*2\.|\n\s*Tempat lahir|\n\s*Tempat\s+lahir)",
        r"Nama lengkap\s*:\s*(.*?)(?:Tempat lahir|Tempat\s+lahir)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            terdakwa = clean_case_value(match.group(1))
            terdakwa = re.sub(r"^\d+\.\s*", "", terdakwa)
            return terdakwa

    # Cadangan dari kalimat amar
    match = re.search(r"Menyatakan\s+Terdakwa\s+(.*?)\s+tersebut", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return clean_case_value(match.group(1))

    return ""


def extract_pengadilan(text):
    match = re.search(r"Pengadilan Negeri\s+Rantau\s*Prapat", text, flags=re.IGNORECASE)
    if match:
        return "PN RANTAU PRAPAT"
    return "PN RANTAU PRAPAT"


def extract_jenis_perkara(text):
    return "Pidana Umum - Penganiayaan"


def extract_pasal(text):
    # Fokus pada pasal terbukti/utama yang menjadi target dataset
    patterns = [
        r"Pasal\s+351\s+Ayat\s*\(1\)\s+KUHPidana",
        r"Pasal\s+351\s+Ayat\s*\(1\)\s+KUHP",
        r"Pasal\s+351\s+ayat\s*\(1\)\s+KUHPidana",
        r"Pasal\s+351\s+ayat\s*\(1\)\s+KUHP",
    ]

    for pattern in patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return "Pasal 351 ayat (1) KUHP"

    # Jika tidak ditemukan, ambil pasal 351 lain sebagai tanda perlu cek
    match = re.search(r"Pasal\s+351\s+ayat\s*\([0-9]\)\s+KUHP", text, flags=re.IGNORECASE)
    if match:
        return clean_case_value(match.group(0))

    return ""


def extract_tanggal_putusan(text):
    # Prioritas: tanggal putusan di bagian akhir setelah frasa diucapkan
    patterns = [
        r"diucapkan\s+dalam\s+sidang\s+terbuka\s+untuk\s+umum\s+pada\s+hari\s+\w+\s+tanggal\s+([0-9]{1,2}\s+\w+\s+[0-9]{4})",
        r"diucapkan.*?pada\s+hari\s+\w+\s+tanggal\s+([0-9]{1,2}\s+\w+\s+[0-9]{4})",
        r"diputuskan.*?pada\s+hari\s+\w+\s+tanggal\s+([0-9]{1,2}\s+\w+\s+[0-9]{4})",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return clean_case_value(match.group(1))

    # Cadangan: ambil tanggal setelah MENGADILI atau tanggal paling akhir
    dates = re.findall(r"([0-9]{1,2}\s+\w+\s+[0-9]{4})", text, flags=re.IGNORECASE)
    if dates:
        return clean_case_value(dates[-1])

    return ""


def extract_amar_putusan(text):
    patterns = [
        r"MENGADILI\s*:\s*(.*?)(?:Demikianlah\s+diputuskan|Hakim\s+Anggota|Panitera\s+Pengganti)",
        r"M E N G A D I L I\s*:\s*(.*?)(?:Demikianlah\s+diputuskan|Hakim\s+Anggota|Panitera\s+Pengganti)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            amar = normalize_space(match.group(1))
            amar = re.sub(r"\n+", " ", amar)
            return clean_case_value(amar)

    return ""


def extract_ringkasan_fakta(text, max_chars=1800):
    # Ambil bagian dakwaan/kronologi yang biasanya diawali dengan "Bahwa Terdakwa..."
    patterns = [
        r"Bahwa\s+Terdakwa\s+(.*?)(?:Perbuatan\s+Terdakwa\s+sebagaimana|Perbuatan\s+terdakwa\s+sebagaimana|Menimbang,\s+bahwa\s+terhadap\s+dakwaan)",
        r"Bahwa\s+terdakwa\s+(.*?)(?:Perbuatan\s+Terdakwa\s+sebagaimana|Perbuatan\s+terdakwa\s+sebagaimana|Menimbang,\s+bahwa\s+terhadap\s+dakwaan)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            fakta = "Bahwa Terdakwa " + match.group(1)
            fakta = normalize_space(fakta)
            fakta = re.sub(r"\n+", " ", fakta)
            return clean_case_value(fakta[:max_chars])

    # Cadangan: ambil fakta hukum
    match = re.search(r"fakta(?:-| )fakta\s+hukum\s+sebagai\s+berikut\s*:\s*(.*?)(?:Menimbang,\s+bahwa\s+selanjutnya|Menimbang\s+bahwa\s+selanjutnya)", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        fakta = normalize_space(match.group(1))
        fakta = re.sub(r"\n+", " ", fakta)
        return clean_case_value(fakta[:max_chars])

    return ""


def detect_label_amar(amar_text):
    """
    Label sederhana berdasarkan amar untuk kebutuhan evaluasi tahap berikutnya.
    """
    text = amar_text.lower()

    if "pidana penjara" in text:
        return "pidana_penjara"
    if "pidana denda" in text:
        return "pidana_denda"
    if "bebas" in text:
        return "bebas"
    if "lepas" in text:
        return "lepas"

    return "lainnya"


## 7. Membaca Seluruh File Teks Putusan

Cell ini membaca seluruh file `.txt` dari folder `data/text/` dan menyimpan isinya ke dalam list untuk diproses menjadi dataset terstruktur.


In [ ]:
txt_files = sorted(TEXT_DIR.glob("*.txt"))

if len(txt_files) == 0:
    raise FileNotFoundError("Tidak ada file TXT di folder data/text/. Pastikan tahap 1 sudah dijalankan.")

print(f"Jumlah file teks yang akan diproses: {len(txt_files)}")
for file in txt_files[:5]:
    print("-", file.name)

## 8. Membuat Representasi Kasus

Cell ini mengekstrak metadata dan konten penting dari setiap putusan. Hasilnya akan menjadi case representation yang digunakan pada tahap retrieval.


In [ ]:
records = []

for txt_file in tqdm(txt_files, desc="Membuat representasi kasus"):
    case_id = txt_file.stem

    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read()

    nomor_perkara = extract_nomor_perkara(text)
    terdakwa = extract_terdakwa(text)
    pengadilan = extract_pengadilan(text)
    jenis_perkara = extract_jenis_perkara(text)
    pasal = extract_pasal(text)
    tanggal_putusan = extract_tanggal_putusan(text)
    amar_putusan = extract_amar_putusan(text)
    ringkasan_fakta = extract_ringkasan_fakta(text)
    label_amar = detect_label_amar(amar_putusan)

    records.append({
        "case_id": case_id,
        "nomor_perkara": nomor_perkara,
        "tanggal_putusan": tanggal_putusan,
        "terdakwa": terdakwa,
        "pengadilan": pengadilan,
        "jenis_perkara": jenis_perkara,
        "pasal": pasal,
        "amar_label": label_amar,
        "amar_putusan": amar_putusan,
        "ringkasan_fakta": ringkasan_fakta,
        "word_count": count_words(text),
        "text_full": text
    })

cases_df = pd.DataFrame(records)
cases_df.head()

## 9. Validasi Kualitas Metadata

Cell ini mengecek apakah metadata penting berhasil diekstrak. Jika ada kolom yang kosong, data tersebut akan ditandai agar dapat diperiksa manual.


In [ ]:
required_columns = [
    "case_id",
    "nomor_perkara",
    "tanggal_putusan",
    "terdakwa",
    "pengadilan",
    "jenis_perkara",
    "pasal",
    "amar_putusan",
    "ringkasan_fakta"
]

quality_records = []

for _, row in cases_df.iterrows():
    missing_columns = []

    for col in required_columns:
        value = str(row.get(col, "")).strip()
        if value == "" or value.lower() == "nan":
            missing_columns.append(col)

    quality_records.append({
        "case_id": row["case_id"],
        "nomor_perkara": row["nomor_perkara"],
        "missing_columns": ", ".join(missing_columns),
        "jumlah_missing": len(missing_columns),
        "status": "valid" if len(missing_columns) == 0 else "perlu_cek"
    })

quality_df = pd.DataFrame(quality_records)

print("Ringkasan kualitas metadata:")
print(quality_df["status"].value_counts())

quality_df

## 10. Mengecek Konsistensi Pasal dan Pengadilan

Cell ini memastikan bahwa seluruh data tetap berada pada domain yang sama, yaitu PN RANTAU PRAPAT, Pidana Umum – Penganiayaan, dan Pasal 351 ayat (1) KUHP.


In [ ]:
print("Distribusi pengadilan:")
display(cases_df["pengadilan"].value_counts())

print("Distribusi jenis perkara:")
display(cases_df["jenis_perkara"].value_counts())

print("Distribusi pasal:")
display(cases_df["pasal"].value_counts())

print("Distribusi label amar:")
display(cases_df["amar_label"].value_counts())

## 11. Preview Representasi Kasus

Cell ini menampilkan beberapa kolom penting agar hasil ekstraksi dapat diperiksa dengan mudah.


In [ ]:
preview_columns = [
    "case_id",
    "nomor_perkara",
    "tanggal_putusan",
    "terdakwa",
    "pasal",
    "amar_label",
    "word_count"
]

cases_df[preview_columns].head(10)

## 12. Menampilkan Contoh Ringkasan Fakta dan Amar Putusan

Cell ini digunakan untuk memeriksa apakah ringkasan fakta dan amar putusan berhasil diambil dari salah satu dokumen.


In [ ]:
sample_index = 0
sample = cases_df.iloc[sample_index]

print("CASE ID:", sample["case_id"])
print("NOMOR PERKARA:", sample["nomor_perkara"])
print("TERDAKWA:", sample["terdakwa"])
print("PASAL:", sample["pasal"])

print("\nRINGKASAN FAKTA:")
print(sample["ringkasan_fakta"][:1500])

print("\nAMAR PUTUSAN:")
print(sample["amar_putusan"][:1500])

## 13. Menyimpan Dataset Terstruktur

Cell ini menyimpan hasil representasi kasus ke dalam format CSV, JSON, dan Excel. File `cases.csv` akan menjadi input utama pada tahap retrieval.


In [ ]:
cases_csv_path = PROCESSED_DIR / "cases.csv"
cases_json_path = PROCESSED_DIR / "cases.json"
cases_xlsx_path = PROCESSED_DIR / "cases.xlsx"
quality_report_path = PROCESSED_DIR / "metadata_quality_report.csv"

cases_df.to_csv(cases_csv_path, index=False)
cases_df.to_json(cases_json_path, orient="records", force_ascii=False, indent=2)
cases_df.to_excel(cases_xlsx_path, index=False)

quality_df.to_csv(quality_report_path, index=False)

print("File berhasil disimpan:")
print("-", cases_csv_path)
print("-", cases_json_path)
print("-", cases_xlsx_path)
print("-", quality_report_path)

## 14. Membuat Ringkasan Tahap 2

Cell ini membuat ringkasan output tahap 2 yang berisi jumlah kasus, jumlah kolom, dan status kualitas metadata.


In [ ]:
stage_02_summary = {
    "jumlah_case": len(cases_df),
    "jumlah_kolom": len(cases_df.columns),
    "jumlah_valid_metadata": int((quality_df["status"] == "valid").sum()),
    "jumlah_perlu_cek": int((quality_df["status"] == "perlu_cek").sum()),
    "output_cases_csv": str(cases_csv_path),
    "output_cases_json": str(cases_json_path),
    "output_quality_report": str(quality_report_path)
}

summary_df = pd.DataFrame([stage_02_summary])
summary_path = PROCESSED_DIR / "stage_02_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary_df

## 15. Membuat ZIP Output Tahap 2

Cell ini membuat ZIP project setelah tahap 2 selesai. ZIP ini dapat diunduh dan digunakan sebagai input untuk tahap berikutnya, yaitu `03_case_retrieval.ipynb`.


In [ ]:
output_zip = Path("/content/stage_02_case_representation_output.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip).replace(".zip", ""),
    format="zip",
    root_dir=BASE_DIR
)

print(f"ZIP output tahap 2 berhasil dibuat: {output_zip}")

## Kesimpulan Tahap 2

Pada tahap ini, seluruh teks putusan dari tahap pertama telah diubah menjadi representasi kasus terstruktur. Dataset `cases.csv` berisi metadata, pasal, amar putusan, ringkasan fakta, dan teks lengkap dari setiap putusan. File ini akan digunakan pada tahap ketiga untuk membangun sistem retrieval menggunakan TF-IDF dan cosine similarity.
